# Real-Time Lakehouse Data Explorer

This notebook allows you to interactively read, inspect, and query data across all layers (**Landing Zone**, **Bronze**, **Silver**, and **Gold**) in real time using **PySpark** and **Delta Lake**.

### 1. Initialize SparkSession & Delta Lake

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in python path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.common.spark_session import get_spark_session
from config.settings import LANDING_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR

# Start or retrieve active SparkSession
spark = get_spark_session(app_name="RealTimeLakehouseExplorer")
spark

### 2. Helper Function to Read Any Layer in Real Time

In [ ]:
def read_table(layer: str = "gold", table_name: str = "enem_performance_by_state"):
    """
    Reads and returns a DataFrame from any specified layer in real time.
    layer: 'landing', 'bronze', 'silver', or 'gold'
    table_name: used when layer='gold'
    """
    if layer == "landing":
        path = LANDING_DIR / "enem_2020" / "MICRODADOS_ENEM_2020.csv"
        return spark.read.option("header", "true").option("sep", ";").option("encoding", "ISO-8859-1").csv(str(path))
    elif layer == "bronze":
        path = BRONZE_DIR / "enem_raw"
        return spark.read.format("delta").load(str(path))
    elif layer == "silver":
        path = SILVER_DIR / "enem_cleaned"
        return spark.read.format("delta").load(str(path))
    elif layer == "gold":
        path = GOLD_DIR / table_name
        return spark.read.format("delta").load(str(path))
    else:
        raise ValueError(f"Invalid layer: {layer}")

### 3. Read Landing Zone (Raw CSV from INEP)

In [ ]:
df_landing = read_table("landing")
print(f"Landing Zone - Total rows: {df_landing.count()} | Total columns: {len(df_landing.columns)}")
df_landing.select("NU_INSCRICAO", "NU_ANO", "SG_UF_PROVA", "TP_SEXO", "NU_NOTA_MT", "NU_NOTA_REDACAO").show(5)

### 4. Read Bronze Layer (Raw Delta Table + Audit Metadata)

In [ ]:
df_bronze = read_table("bronze")
print(f"Bronze Layer - Total rows: {df_bronze.count()}")
# Notice the audit columns automatically added by the pipeline: _ingestion_timestamp and _source_file
df_bronze.select("NU_INSCRICAO", "NU_ANO", "SG_UF_PROVA", "_ingestion_timestamp", "_source_file").show(5, truncate=False)

### 5. Read Silver Layer (Cleaned, Standardized, Typed, & Deduplicated)

In [ ]:
df_silver = read_table("silver")
print(f"Silver Layer - Total rows: {df_silver.count()} | Columns: {len(df_silver.columns)}")
# Notice clean snake_case column names and decimal score types
df_silver.select("registration_id", "year", "exam_state", "score_math", "score_essay", "score_natural_sciences").show(5)

### 6. Read Gold Layer (Curated Data Marts)

In [ ]:
# Data Mart 1: Academic Performance by State
df_gold_state = read_table("gold", "enem_performance_by_state")
df_gold_state.show(truncate=False)

In [ ]:
# Data Mart 2: Performance by School Type (Public vs. Private)
df_gold_school = read_table("gold", "enem_performance_by_school")
df_gold_school.show(10, truncate=False)

In [ ]:
# Data Mart 3: Candidate Fact Table with Performance Tiers
df_gold_fact = read_table("gold", "enem_fact_candidates")
df_gold_fact.select("registration_id", "exam_state", "average_score", "performance_tier").show(10)

### 7. Interactive SQL Queries in Real Time
You can register any DataFrame as a temporary view and query it with standard SQL!

In [ ]:
df_silver.createOrReplaceTempView("silver_enem")

# Query top students with math score >= 700
spark.sql("""
    SELECT registration_id, exam_state, gender, score_math, score_essay
    FROM silver_enem
    WHERE score_math >= 700
    ORDER BY score_math DESC
""").show()

### 8. Delta Lake Transaction Log & History (Time Travel)

In [ ]:
# Inspect Delta Lake ACID commit history in real time
spark.sql("DESCRIBE HISTORY delta.`data/silver/enem_cleaned`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

### 9. Close Spark Session When Done

In [ ]:
spark.stop()
print("SparkSession closed successfully.")